<a href="https://colab.research.google.com/github/syedebrah/CodeBase/blob/main/Qwen2_5_Coder_0_5B_Instruct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
# Prevent PyTorch memory fragmentation just to be safe
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# 1. Install Unsloth and Dependencies (Notice 'trl' is no longer restricted!)
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers peft accelerate bitsandbytes datasets wandb
!pip install -U trl

In [ ]:
  # 2. Authenticate Hugging Face
from huggingface_hub import login
print("Login to Hugging Face:")
login() # Paste your write-access token here

In [ ]:
# 3. Authenticate Weights & Biases
import wandb
print("\nLogin to Weights & Biases:")
wandb.login() # Paste your API key here

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = False

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-0.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 2. Attach LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# 1. Standardize formatting to ChatML
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
    mapping = {"role" : "role", "content" : "content", "user" : "user", "assistant" : "assistant"},
    map_eos_token = True,
)

# 2. Load and filter the data
dataset = load_dataset("nampdn-ai/tiny-codes", split="train")
js_dataset = dataset.filter(lambda row: row["programming_language"].lower() == "javascript")

# 3. Apply the conversational template
def format_dataset(examples):
    queries = examples["prompt"]
    codes = examples["response"]

    formatted_texts = []
    for query, code in zip(queries, codes):
        conversation = [
            {"role": "user", "content": query},
            {"role": "assistant", "content": f"```javascript\n{code}\n```"}
        ]
        text = tokenizer.apply_chat_template(
            conversation, tokenize=False, add_generation_prompt=False
        )
        formatted_texts.append(text)
    return {"text": formatted_texts}

js_dataset = js_dataset.map(format_dataset, batched=True)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
import os
import wandb

# 1. Initialize the JSSLM tracking project in WandB
run_name = "qwen0-run1"
wandb.init(project="Qwen", name=run_name)
os.environ["WANDB_LOG_MODEL"] = "checkpoint"

# 2. Configure the Trainer
trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = js_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "wandb",
        run_name = run_name,

        dataset_text_field = "text",
        max_length = max_seq_length,
        dataset_num_proc = 2,
        packing = False,
    ),
)

# 3. Start Training!
trainer_stats = trainer.train()

# 4. Close WandB gracefully
wandb.finish()

In [ ]:
from transformers import TextStreamer

# Enable 2x faster native inference
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "Write a JavaScript Code to allow voters based on the Age ."},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt = True)

print("--- Generating Pure JavaScript ---")
_ = model.generate(
    input_ids = inputs,
    streamer = text_streamer,
    max_new_tokens = 512,
    use_cache = True,
    temperature = 0.1,
)

In [ ]:
from transformers import TextStreamer

# Enable 2x faster native inference
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "Write a JavaScript function that takes an array of objects and sorts them by age in descending order."},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt = True)

print("--- Generating Pure JavaScript ---")
_ = model.generate(
    input_ids = inputs,
    streamer = text_streamer,
    max_new_tokens = 512,
    use_cache = True,
    temperature = 0.1,
)

In [ ]:
from transformers import TextStreamer

# Enable 2x faster native inference
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "Write a JavaScript snippet that selects a button with the ID 'theme-toggle' and adds a click event listener that toggles a dark-mode class on the document body."},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt = True)

print("--- Generating Pure JavaScript ---")
_ = model.generate(
    input_ids = inputs,
    streamer = text_streamer,
    max_new_tokens = 512,
    use_cache = True,
    temperature = 0.1,
)

In [ ]:
from transformers import TextStreamer

# Enable 2x faster native inference
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "Write an asynchronous JavaScript function that fetches data from a 'students.json' file. Once the JSON is parsed, use the array filter method to return only the records where the 'department' property is exactly 'BCA' and the 'year' property is exactly 1."},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt = True)

print("--- Generating Pure JavaScript ---")
_ = model.generate(
    input_ids = inputs,
    streamer = text_streamer,
    max_new_tokens = 1024,
    use_cache = True,
    temperature = 0.1,
)

In [ ]:
from transformers import TextStreamer

# Enable 2x faster native inference
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "Write a frontend JavaScript function using the browser's native Fetch API to retrieve data from the 'students.json' endpoint. Once the JSON is parsed, use the array filter method to return only the records where the 'department' property is exactly 'BCA' and the 'year' property is exactly 1. Include a try/catch block for error handling."},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt = True)

print("--- Generating Pure JavaScript ---")
_ = model.generate(
    input_ids = inputs,
    streamer = text_streamer,
    max_new_tokens = 1024,
    use_cache = True,
    temperature = 0.1,
)

In [ ]:
messages = [
    {"role": "user", "content": "Write a frontend JavaScript function using the browser's native Fetch API to retrieve 'students.json'. Filter and return only the records where department is 'BCA' and year is 1. Do not use require or imports. Start the code immediately."},
]

# 1. Apply the template but keep it as a string
prompt_string = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,
)

# 2. THE TRICK: Force the model to start inside the function!
prompt_string += "```javascript\nasync function getBCAStudents() {\n"

# 3. Tokenize and send to GPU
inputs = tokenizer([prompt_string], return_tensors = "pt").to("cuda")

print("--- Generating Pure Browser JavaScript ---")
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 512,
    use_cache = True,
    temperature = 0.1,
)